# Codet5 full fine

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, RobertaTokenizer, AdamW, get_linear_schedule_with_warmup
import torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                            roc_auc_score, matthews_corrcoef, cohen_kappa_score, 
                            mean_squared_error, mean_absolute_error, confusion_matrix)
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class CodeT5Classifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super(CodeT5Classifier, self).__init__()
        self.t5_model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.t5_model.config.d_model, num_labels)
        
    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.t5_model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        hidden_states = encoder_outputs.last_hidden_state
        pooled_output = hidden_states[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

class CodeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []
    
    progress_bar = tqdm(dataloader, desc='Training')
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    all_probs = []
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc='Evaluating')
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted')
    
    return avg_loss, accuracy, precision, recall, f1, predictions, true_labels, all_probs

def calculate_metrics(true_labels, predictions, all_probs, num_classes):
    from sklearn.preprocessing import label_binarize
    
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted')
    
    y_true_bin = label_binarize(true_labels, classes=list(range(num_classes)))
    if num_classes == 2:
        auc = roc_auc_score(true_labels, np.array(all_probs)[:, 1])
    else:
        try:
            auc = roc_auc_score(y_true_bin, all_probs, multi_class='ovr', average='weighted')
        except:
            auc = 0.0
    
    mcc = matthews_corrcoef(true_labels, predictions)
    kappa = cohen_kappa_score(true_labels, predictions)
    
    mse = mean_squared_error(true_labels, predictions)
    mae = mean_absolute_error(true_labels, predictions)
    
    cm = confusion_matrix(true_labels, predictions)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': int(tp),
        'FN': int(fn)
    }

def main():
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    
    print(f"Train dataset size: {len(train_df)}")
    print(f"Test dataset size: {len(test_df)}")
    print(f"Label distribution in train: {train_df['label'].value_counts().sort_index().to_dict()}")
    print(f"Label distribution in test: {test_df['label'].value_counts().sort_index().to_dict()}\n")
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    print("Loading CodeT5 tokenizer and model...")
    model_name = "Salesforce/codet5-base"
    tokenizer = RobertaTokenizer.from_pretrained(model_name)
    model = CodeT5Classifier(model_name, num_labels=6)
    model.to(device)
    
    print("Creating datasets and dataloaders...")
    train_dataset = CodeDataset(
        train_df['func'].values,
        train_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    test_dataset = CodeDataset(
        test_df['func'].values,
        test_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    batch_size = 8
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    epochs = 5
    learning_rate = 2e-5
    
    optimizer = AdamW(model.parameters(), lr=learning_rate, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()
    
    print(f"\nStarting training for {epochs} epochs...")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print(f"Total training steps: {total_steps}\n")
    
    best_accuracy = 0
    
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        print("-" * 50)
        
        train_loss, train_accuracy = train_epoch(model, train_dataloader, optimizer, scheduler, criterion, device)
        print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
        
        test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels, all_probs = evaluate(
            model, test_dataloader, criterion, device
        )
        print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Precision: {test_precision:.4f} | Test Recall: {test_recall:.4f} | Test F1: {test_f1:.4f}\n")
        
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            torch.save(model.state_dict(), 'best_codet5_model.pt')
            print(f"Best model saved with accuracy: {best_accuracy:.4f}\n")
    
    print("Training completed!")
    print("=" * 80)
    print("\nFinal Test Results:")
    print("=" * 80)
    
    model.load_state_dict(torch.load('best_codet5_model.pt'))
    test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels, all_probs = evaluate(
        model, test_dataloader, criterion, device
    )
    
    metrics = calculate_metrics(true_labels, predictions, all_probs, num_classes=6)
    
    print("\nComprehensive Classification Metrics:")
    print("-" * 80)
    print(f"{'AUC':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'MCC':<12} {'Kappa':<12} {'MSE':<12} {'MAE':<12} {'TP':<12} {'FN':<12}")
    print(f"{metrics['AUC']:<12.4f} {metrics['Accuracy']:<12.4f} {metrics['Precision']:<12.4f} {metrics['Recall']:<12.4f} "
          f"{metrics['F1']:<12.4f} {metrics['MCC']:<12.4f} {metrics['Kappa']:<12.4f} {metrics['MSE']:<12.4f} "
          f"{metrics['MAE']:<12.4f} {metrics['TP']:<12} {metrics['FN']:<12}")
    
    print("\n" + "=" * 80)
    print("\nDetailed Classification Report:")
    print("=" * 80)
    print(classification_report(true_labels, predictions, digits=4))
    
    print("\nConfusion Matrix:")
    print("-" * 80)
    cm = confusion_matrix(true_labels, predictions)
    print(cm)
    
    print("\n" + "=" * 80)
    print("Model saved as 'best_codet5_model.pt'")
    print("=" * 80)

if __name__ == "__main__":
    main()

Loading datasets...
Train dataset size: 21793
Test dataset size: 5400
Label distribution in train: {0: 3800, 1: 3566, 2: 3640, 3: 3575, 4: 3591, 5: 3621}
Label distribution in test: {0: 900, 1: 934, 2: 860, 3: 918, 4: 909, 5: 879}

Device: mps

Loading CodeT5 tokenizer and model...
Creating datasets and dataloaders...

Starting training for 5 epochs...
Batch size: 8
Learning rate: 2e-05
Total training steps: 13625

Epoch 1/5
--------------------------------------------------


Training: 100%|██████████████████| 2725/2725 [59:37<00:00,  1.31s/it, loss=2.94]


Train Loss: 0.9475 | Train Accuracy: 0.6492


Evaluating: 100%|█████████████████████████████| 675/675 [04:25<00:00,  2.55it/s]


Test Loss: 1.1774 | Test Accuracy: 0.6124
Test Precision: 0.6866 | Test Recall: 0.6124 | Test F1: 0.5762

Best model saved with accuracy: 0.6124

Epoch 2/5
--------------------------------------------------


...

Train Loss: 0.2814 | Train Accuracy: 0.9118


Evaluating: 100%|█████████████████████████████| 675/675 [02:40<00:00,  4.21it/s]


Test Loss: 1.0430 | Test Accuracy: 0.7431
Test Precision: 0.7621 | Test Recall: 0.7431 | Test F1: 0.7459

Best model saved with accuracy: 0.7431

Training completed!

Final Test Results:


Evaluating: 100%|█████████████████████████████| 675/675 [02:42<00:00,  4.15it/s]


Comprehensive Classification Metrics:
--------------------------------------------------------------------------------
AUC          Accuracy     Precision    Recall       F1           MCC          Kappa        MSE          MAE          TP           FN          
0.9352       0.7431       0.7621       0.7431       0.7459       0.6936       0.6917       1.9037       0.6237       4013         1387        


Detailed Classification Report:
              precision    recall  f1-score   support

           0     0.9967    0.6644    0.7973       900
           1     0.5387    0.6188    0.5760       934
           2     0.7454    0.8512    0.7948       860
           3     0.9260    0.9270    0.9265       918
           4     0.7307    0.7789    0.7540       909
           5     0.6371    0.6212    0.6290       879

    accuracy                         0.7431      5400
   macro avg     0.7624    0.7436    0.7463      5400
weighted avg     0.7621    0.7431    0.7459      5400


Confusion Matrix

# unixcoder fine tuning

In [ ]:

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                            roc_auc_score, matthews_corrcoef, cohen_kappa_score, 
                            mean_squared_error, mean_absolute_error, confusion_matrix)
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class UniXcoderClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super(UniXcoderClassifier, self).__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[0][:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

class CodeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []
    
    progress_bar = tqdm(dataloader, desc='Training')
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    all_probs = []
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc='Evaluating')
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted')
    
    return avg_loss, accuracy, precision, recall, f1, predictions, true_labels, all_probs

def calculate_metrics(true_labels, predictions, all_probs, num_classes):
    from sklearn.preprocessing import label_binarize
    
    acc = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted')
    
    y_true_bin = label_binarize(true_labels, classes=list(range(num_classes)))
    if num_classes == 2:
        auc = roc_auc_score(true_labels, np.array(all_probs)[:, 1])
    else:
        try:
            auc = roc_auc_score(y_true_bin, all_probs, multi_class='ovr', average='weighted')
        except:
            auc = 0.0
    
    mcc = matthews_corrcoef(true_labels, predictions)
    kappa = cohen_kappa_score(true_labels, predictions)
    
    mse = mean_squared_error(true_labels, predictions)
    mae = mean_absolute_error(true_labels, predictions)
    
    cm = confusion_matrix(true_labels, predictions)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    
    return {
        'AUC': auc,
        'Acc': acc,
        'Pre': precision,
        'Rec': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': tp,
        'FN': fn
    }

def main():
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    

    
    print(f"Train dataset size: {len(train_df)}")
    print(f"Test dataset size: {len(test_df)}")
    print(f"Label distribution in train: {train_df['label'].value_counts().sort_index().to_dict()}")
    print(f"Label distribution in test: {test_df['label'].value_counts().sort_index().to_dict()}\n")
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    print("Loading UniXcoder tokenizer and model...")
    model_name = "microsoft/unixcoder-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = UniXcoderClassifier(model_name, num_labels=6)
    model.to(device)
    
    print("Creating datasets and dataloaders...")
    train_dataset = CodeDataset(
        train_df['func'].values,
        train_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    test_dataset = CodeDataset(
        test_df['func'].values,
        test_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    batch_size = 8
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    epochs = 5
    learning_rate = 2e-5
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=0.0,
        total_iters=total_steps
    )
    criterion = nn.CrossEntropyLoss()
    
    print(f"\nStarting training for {epochs} epochs...")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print(f"Total training steps: {total_steps}\n")
    
    best_accuracy = 0
    
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        print("-" * 50)
        
        train_loss, train_accuracy = train_epoch(model, train_dataloader, optimizer, scheduler, criterion, device)
        print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
        
        test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels, all_probs = evaluate(
            model, test_dataloader, criterion, device
        )
        print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Precision: {test_precision:.4f} | Test Recall: {test_recall:.4f} | Test F1: {test_f1:.4f}\n")
        
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            torch.save(model.state_dict(), 'best_unixcoder_model.pt')
            print(f"Best model saved with accuracy: {best_accuracy:.4f}\n")
    
    print("Training completed!")
    print("=" * 80)
    print("\nFinal Test Results:")
    print("=" * 80)
    
    model.load_state_dict(torch.load('best_unixcoder_model.pt'))
    test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels, all_probs = evaluate(
        model, test_dataloader, criterion, device
    )
    
    metrics = calculate_metrics(true_labels, predictions, all_probs, num_classes=6)
    
    print("\nComprehensive Classification Metrics:")
    print("-" * 80)
    print(f"{'AUC':<10} {'Acc':<10} {'Pre':<10} {'Rec':<10} {'F1':<10} {'MCC':<10} {'Kappa':<10} {'MSE':<10} {'MAE':<10} {'TP':<10} {'FN':<10}")
    print(f"{metrics['AUC']:<10.4f} {metrics['Acc']:<10.4f} {metrics['Pre']:<10.4f} {metrics['Rec']:<10.4f} "
          f"{metrics['F1']:<10.4f} {metrics['MCC']:<10.4f} {metrics['Kappa']:<10.4f} {metrics['MSE']:<10.4f} "
          f"{metrics['MAE']:<10.4f} {metrics['TP']:<10} {metrics['FN']:<10}")
    
    print("\n" + "=" * 80)
    print("\nDetailed Classification Report:")
    print("=" * 80)
    print(classification_report(true_labels, predictions, digits=4))
    
    print("\nConfusion Matrix:")
    print("-" * 80)
    cm = confusion_matrix(true_labels, predictions)
    print(cm)

if __name__ == "__main__":
    main()

# santacoder full fine

In [ ]:

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, GPTBigCodeForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class CodeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []
    
    progress_bar = tqdm(dataloader, desc='Training')
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
   
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc='Evaluating')
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted')
    
    return avg_loss, accuracy, precision, recall, f1, predictions, true_labels

def main():
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    
    print(f"Train dataset size: {len(train_df)}")
    print(f"Test dataset size: {len(test_df)}")
    print(f"Label distribution in train: {train_df['label'].value_counts().sort_index().to_dict()}")
    print(f"Label distribution in test: {test_df['label'].value_counts().sort_index().to_dict()}\n")
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    print("Loading SantaCoder tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained('bigcode/santacoder', trust_remote_code=True)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = GPTBigCodeForSequenceClassification.from_pretrained(
        'bigcode/santacoder',
        num_labels=6,
        ignore_mismatched_sizes=True,
        trust_remote_code=True
    )
    
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    
    model.to(device)
    
    print("Creating datasets and dataloaders...")
    train_dataset = CodeDataset(
        train_df['func'].values,
        train_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    test_dataset = CodeDataset(
        test_df['func'].values,
        test_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    batch_size = 8
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    epochs = 5
    learning_rate = 5e-5
    
    optimizer = AdamW(model.parameters(), lr=learning_rate, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    print(f"\nStarting training for {epochs} epochs...")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print(f"Total training steps: {total_steps}\n")
    
    best_accuracy = 0
    
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        print("-" * 50)
        
        train_loss, train_accuracy = train_epoch(model, train_dataloader, optimizer, scheduler, device)
        print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
        
        test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels = evaluate(
            model, test_dataloader, device
        )
        print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Precision: {test_precision:.4f} | Test Recall: {test_recall:.4f} | Test F1: {test_f1:.4f}\n")
        
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            torch.save(model.state_dict(), 'best_santacoder_model.pt')
            print(f"Best model saved with accuracy: {best_accuracy:.4f}\n")
    
    print("Training completed!")
    print("=" * 50)
    print("\nFinal Test Results:")
    print("=" * 50)
    
    model.load_state_dict(torch.load('best_santacoder_model.pt'))
    test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels = evaluate(
        model, test_dataloader, device
    )
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall: {test_recall:.4f}")
    print(f"Test F1 Score: {test_f1:.4f}\n")
    
    print("Classification Report:")
    print(classification_report(true_labels, predictions, digits=4))

if __name__ == "__main__":
    main()

# codebert training

In [ ]:

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class CodeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []
    
    progress_bar = tqdm(dataloader, desc='Training')
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc='Evaluating')
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted')
    
    return avg_loss, accuracy, precision, recall, f1, predictions, true_labels

def main():
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    

    print(f"Train dataset size: {len(train_df)}")
    print(f"Test dataset size: {len(test_df)}")
    print(f"Label distribution in train: {train_df['label'].value_counts().sort_index().to_dict()}")
    print(f"Label distribution in test: {test_df['label'].value_counts().sort_index().to_dict()}\n")
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    print("Loading CodeBERT tokenizer and model...")
    tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
    model = RobertaForSequenceClassification.from_pretrained(
        'microsoft/codebert-base',
        num_labels=6
    )
    model.to(device)
    
    print("Creating datasets and dataloaders...")
    train_dataset = CodeDataset(
        train_df['func'].values,
        train_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    test_dataset = CodeDataset(
        test_df['func'].values,
        test_df['label'].values,
        tokenizer,
        max_length=512
    )
    
    batch_size = 8
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    epochs = 5
    learning_rate = 2e-5
    
    optimizer = AdamW(model.parameters(), lr=learning_rate, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    print(f"\nStarting training for {epochs} epochs...")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print(f"Total training steps: {total_steps}\n")
    
    best_accuracy = 0
    
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        print("-" * 50)
        
        train_loss, train_accuracy = train_epoch(model, train_dataloader, optimizer, scheduler, device)
        print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
        
        test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels = evaluate(
            model, test_dataloader, device
        )
        print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Precision: {test_precision:.4f} | Test Recall: {test_recall:.4f} | Test F1: {test_f1:.4f}\n")
        
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            torch.save(model.state_dict(), 'best_codebert_model.pt')
            print(f"Best model saved with accuracy: {best_accuracy:.4f}\n")
    
    print("Training completed!")
    print("=" * 50)
    print("\nFinal Test Results:")
    print("=" * 50)
    
    model.load_state_dict(torch.load('best_codebert_model.pt'))
    test_loss, test_accuracy, test_precision, test_recall, test_f1, predictions, true_labels = evaluate(
        model, test_dataloader, device
    )
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall: {test_recall:.4f}")
    print(f"Test F1 Score: {test_f1:.4f}\n")
    
    print("Classification Report:")
    print(classification_report(true_labels, predictions, digits=4))

if __name__ == "__main__":
    main()
